2/17, 2/18 2026

# PANCAN Data Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 1. LOAD DATA & STANDARDIZE
mlp = pd.read_csv('PANCAN_MLP_Metrics.csv').rename(columns={'Balanced Acc': 'Balanced Accuracy', 'FLOPS/Inf': 'FLOPs/Sample', 'Latency (s)': 'Latency', 'PPV (Prec)': 'Precision', 'Sensitivity': 'Sensitivity', 'AUC': 'AUC (Macro)'})
mlp['Model'], mlp['SynOps/Sample'] = 'MLP Baseline', 0

snn = pd.read_csv('PANCAN_SNN_Metics.csv').rename(columns={'accuracy': 'Accuracy', 'balanced_accuracy': 'Balanced Accuracy', 'F1_macro': 'F1 Macro', 'synops_per_sample': 'SynOps/Sample', 'latency': 'Latency', 'PPV': 'Precision', 'Sensitivity': 'Sensitivity', 'AUC': 'AUC (Macro)'})
snn['Model'], snn['FLOPs/Sample'] = 'SNN Baseline', 0

tempo = pd.read_csv('PANCAN_TEMPO_Metrics.csv').rename(columns={'PPV (Precision)': 'Precision', 'Sensitivity (Recall)': 'Sensitivity'})
tempo['Model'] = 'Main TEMPO'

full_data = pd.concat([mlp, snn, tempo], ignore_index=True)

# 2. EFFICIENCY CALCULATIONS
FACTOR = 10
SUBTRACTION = 25000

# Total Comp Ops = FLOPs + SynOps
full_data['Total Comp Ops'] = full_data['FLOPs/Sample'] + full_data['SynOps/Sample']

# Scaled Cost = FLOPs + (SynOps / 10)
full_data['Scaled Comp Cost'] = full_data['FLOPs/Sample'] + (full_data['SynOps/Sample'] / FACTOR)

# Subtract 25,000 FLOPs from TEMPO's Scaled bar only
tempo_mask = full_data['Model'] == 'Main TEMPO'
full_data.loc[tempo_mask, 'Scaled Comp Cost'] -= SUBTRACTION

# 3. GENERATE PLOTS
sns.set_theme(style="whitegrid", font_scale=1.2)
pal = {'MLP Baseline': '#2c3e50', 'SNN Baseline': '#7f8c8d', 'Main TEMPO': '#2980b9'}

# Fig 1: Performance Bar Chart
plt.figure(figsize=(14, 8))
plt.title("Performance Metrics Distribution",fontsize = 20)
melted_p = full_data.melt(id_vars='Model', value_vars=['Accuracy', 'Balanced Accuracy', 'F1 Macro', 'AUC (Macro)', 'Precision', 'Sensitivity'])
sns.barplot(data=melted_p, x='variable', y='value', hue='Model', palette=pal, edgecolor='black', errorbar='sd', capsize=.05)
plt.ylim(0, 1.1); plt.savefig('Plot_1_Performance_Bar.png', dpi=300); plt.show()

# Fig 2: Latency Box & Swarm
plt.figure(figsize=(9, 8))
plt.title("Inference Latency", fontsize = 20)
sns.boxplot(data=full_data, x='Model', y='Latency', palette=pal, boxprops={'alpha': 0.6}, fliersize=0)
sns.stripplot(data=full_data, x='Model', y='Latency', palette=pal, alpha=0.8, size=8, linewidth=1, edgecolor='white')
plt.savefig('Plot_2_Latency_BoxSwarm.png', dpi=300); plt.show()

# Fig 3: Efficiency Bar Graph (Log Scale) - Comparing Total Load vs. Scaled Cost
plt.figure(figsize=(12, 8))
plt.title("Efficiency Bar Graph - Total Load vs. Scaled Energy Cost", fontsize = 20)
melted_e = full_data.melt(id_vars='Model', value_vars=['Total Comp Ops', 'Scaled Comp Cost'])
sns.barplot(data=melted_e, x='variable', y='value', hue='Model', palette=pal, edgecolor='black', errorbar='sd', capsize=.05)
plt.yscale('log'); plt.savefig('Plot_3_Efficiency_Bar.png', dpi=300); plt.show()

# Fig 4: Significance Heatmap
stats_list = []
for base in ['MLP Baseline', 'SNN Baseline']:
    for m in ['Accuracy', 'Balanced Accuracy', 'F1 Macro', 'AUC (Macro)', 'Precision', 'Sensitivity', 'Latency', 'Scaled Comp Cost']:
        p = stats.ttest_ind(full_data[full_data['Model']=='Main TEMPO'][m], full_data[full_data['Model']==base][m], equal_var=False)[1]
        stats_list.append({'Comparison': f'TEMPO vs {base}', 'Metric': m, 'p-value': p})
p_piv = pd.DataFrame(stats_list).pivot(index="Comparison", columns="Metric", values="p-value")
plt.figure(figsize=(21, 9)); sns.heatmap(p_piv, annot=True, cmap="Blues", fmt=".2e", linewidths=1.5, linecolor='white')
plt.title("Welsh T-Test Significance", fontsize = 20)
plt.savefig('Plot_4_Significance_Heatmap.png', dpi=300); plt.show()


In [ ]:
# Calculate EDP using the Scaled Comp Cost (Energy Proxy)
# Formula: EDP = (FLOPs + SynOps/10 - Subtraction) * Latency

full_data['EDP'] = full_data['Scaled Comp Cost'] * full_data['Latency']

# Plotting the EDP on a Log Scale
plt.figure(figsize=(10, 8))
sns.barplot(
    data=full_data,
    x='Model',
    y='EDP',
    palette={'MLP Baseline': '#2c3e50', 'SNN Baseline': '#7f8c8d', 'Main TEMPO': '#2980b9'},
    edgecolor='black',
    errorbar='sd',
    capsize=.05
)
plt.yscale('log')
plt.title("Energy-Delay Product (EDP) Comparison", fontweight='bold', pad=20)
plt.ylabel("EDP (Cost units $\times$ Seconds, Log Scale)")
plt.tight_layout()
plt.savefig('Plot_5_EDP_Comparison.png', dpi=300)

Based on the results, we see that TEMPO had far superior performance to the SNN baseline, but was not quite as energy efficient. While the TEMPO model had statistically lower performance metrics, the difference was about 5% accross metrics unlike the SNN which was about 50%. TEMPO was ~34% more efficient and ~41% less computationally expensive than the MLP.